# CS781 Phase 7 — BiLSTM RETRIEVER: implementation, a new Kaggle dataset that stores the model, and the predictions

**What this notebook is.** The fourth Phase 1 architecture (the BiLSTM) as a prior-case retriever, end to end: (1) the BiLSTM retriever (a bi-encoder: every 512-token chunk -> the mean of the Phase 1 BiLSTM's forward and backward states -> a unit vector; document score = mean-pooled cosine or chunk MaxSim, chosen on dev), trained contrastively (in-batch + BM25 hard negatives; other relevant candidates masked);
(2) a NEW KAGGLE DATASET that stores the trained model (verified upload); (3) the Phase 7 predictions: `prediction_bilstm_only.json` and `prediction.json` (+ zips with the file at the root), 7 unique candidates per test query from the TEST pool.

| step | what | time (T4, estimate) |
|---|---|---|
| data, tokenise, BM25 | the six Phase 7 files (schema detected, each split with its own pool), Phase 1 tokenizer, BM25 baseline and hard negatives | 10 min |
| zero-shot | the untouched Phase 1 BiLSTM as a retriever: how well do its representations transfer? | 8-12 min |
| continued pre-training (optional, `DAPT_ON`) | the Phase 1 BiLSTM continues its own bidirectional LM objective on the TRAIN documents of Phase 7 (provided text only); a sanity check first (the loss must look like the Phase 1 loss); only USED if it helps on dev | 40 min |
| **BL1 / BL3 / BL2 / BL4** | partial fine-tuning (top half of the LSTM layers) and full fine-tuning, each from the Phase 1 weights and from the continued-pretrained weights (LoRA, LoRMA and IA3 act on linear layers; an LSTM has none) | 4 x (30 + 8-12) min |
| ensemble, fusion | the BiLSTM runs averaged; if the output of the ModernBERT notebooks (`pcr/`) is attached, fused with its final system (adopted only if dev rises by 0.005 with a paired interval above 0) | seconds |
| dataset | checkpoints, scores, `arch_loaders.py`, manifest, README -> `kaggle datasets create`, then the status is polled | 3-10 min |
| predictions | the best BiLSTM system writes the files; in predict-only mode the saved checkpoint is re-encoded and compared with the stored scores | seconds - 12 min |
Total about 3-3.5 hours (`TIME_PROFILE = "fast"`: about 1.5 hours); a time budget skips any step that would not fit, and `prediction.json` is rewritten after every run.

## Predict-only mode
Attach the dataset this notebook creates (or an earlier output of it) and run with `RETRAIN = False`: nothing is trained; the files are restored, the best checkpoint is re-encoded and the predictions are written again.

## Inputs
The **BiLSTM Phase 1 dataset** (`cs781-phase1-v3-bilstm-recurrent`: it holds `final_model.pt`; without it the notebook stops with a clear message unless a retriever dataset is attached), a Phase 1 dataset that holds the tokenizer (any of them), Secrets `HF_TOKEN` and, for the upload, `KAGGLE_USERNAME` and `KAGGLE_KEY`, Internet ON, a GPU. Optional: the output of the ModernBERT notebooks (folder `pcr/`).

## Rules
Starts from the Phase 1 BiLSTM; no outside model, weights, corpora or APIs; no test label; no dev or test text in the continued pre-training. Not a counted experiment of the ModernBERT notebooks: Phase 7 does not require all four architectures.
## Honest expectation
The BiLSTM was pre-trained for only 560 steps (validation loss 5.96 against 6.56 for a word-frequency baseline) and sees only 827 training queries: expect it to be clearly weaker than the ModernBERT retriever (dev micro-F1@7 0.35 there) and perhaps near BM25. Its value is the fourth architecture, the transfer analysis and possible ensemble diversity. **Tested** on CPU with tiny models and synthetic data (a stand-in `kaggle` command for the upload), including predict-only mode from the created dataset; not on the real data or a GPU. All timings are estimates.


In [ ]:
# ================= CELL 0 — CONFIG  [RUNS NOW]. Edit this cell only. =================
HF_DATASET = "Exploration-Lab/CS-781-Capstone"; DATA_SUBDIR = "Phase_7_PCR"                     # the gated dataset and its Phase 7 folder
DATA_SOURCE = "hub_files"                                                                      # "hub_files" = download the six Phase 7 json files; "files" = read them from LOCAL_DATA_DIR
LOCAL_DATA_DIR = None
CKPT_PATHS = {"gpt2_new": None, "modernbert": None, "t5": None, "bilstm": None}                # Phase 1 final_model.pt paths; None = found automatically under /kaggle/input. ONLY THE BiLSTM IS USED: attach the dataset cs781-phase1-v3-bilstm-recurrent
TOKENIZER_PATH = None
LSI_WORK = "/kaggle/working/pcr_bilstm"                                                        # everything this notebook writes (a different folder from Parts 1-2 of the ModernBERT notebooks)
TOP_K = 7
TIME_PROFILE = "normal"                                                                        # "normal" (about 3-3.5 h) or "fast" (about 1.5 h)
DAPT_ON = True                                                                                 # continued pre-training of the Phase 1 BiLSTM on the TRAIN documents of Phase 7 (provided text only; no external corpus). It is only USED if it helps on dev. False = skip it.
UPLOAD_DATASET = True                                                                          # create the Kaggle dataset that stores the trained model (needs the KAGGLE_USERNAME and KAGGLE_KEY secrets)
DATASET_SLUG, DATASET_TITLE = "cs781-phase7-bilstm-retriever", "CS781 Phase 7 BiLSTM retriever"
RETRAIN = False                                                                                # False = if the retriever dataset (or an earlier output of this notebook) is attached, do NOT train again: only regenerate the predictions
REGENERATE_FROM_CHECKPOINT = True                                                              # in that predict-only case also re-encode the test pool with the saved checkpoint (checks that the saved model really reproduces the scores)
RULE_MODE = "strict"
RERUN = False
SELFTEST = False
NB_T0 = __import__("time").time()


In [ ]:
# ================= CELL 1 — setup (Phase 7: BiLSTM retriever) =================
import os, sys, re, json, math, time, random, glob, gc, copy, hashlib, pickle, getpass, zipfile, shutil
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); AMP = device.type == "cuda"
WORK = Path(LSI_WORK); (WORK / "runs").mkdir(parents=True, exist_ok=True); (WORK / "cache").mkdir(exist_ok=True); SUBDIR = WORK / "submissions"; SUBDIR.mkdir(exist_ok=True)
LEDGER = WORK / "ledger.csv"
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working"]
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu (training cells need a GPU)", "| work dir:", WORK)


In [ ]:
# ===== CREDENTIALS CHECK  [RUNS NOW; it never asks you to type anything]: reads HF_TOKEN (and optionally KAGGLE_USERNAME / KAGGLE_KEY) from Kaggle Secrets, verifies them, and stops early with the exact fix =====
# Needs Internet ON. Add the Secrets under Add-ons > Secrets with 'Attached' ON for THIS notebook (a committed run only sees attached secrets). The Hugging Face token is not needed when DATA_SOURCE = "files".
import os, subprocess, shutil
REQUIRE_HF, REQUIRE_KAGGLE = (DATA_SOURCE in ("hf", "hub_files")), UPLOAD_DATASET        # the Kaggle key is only needed for the optional backup upload (set REQUIRE_KAGGLE = True to insist)
def get_secret(name):
    v = os.environ.get(name)
    if not v:
        try:
            from kaggle_secrets import UserSecretsClient; v = UserSecretsClient().get_secret(name)
        except Exception: v = None
    return (v or "").strip().strip("'\"") or None
problems, report = [], []
hf = get_secret("HF_TOKEN"); HF_OK = False
if DATA_SOURCE not in ("hf", "hub_files"): report.append("Hugging Face token : not needed (DATA_SOURCE = 'files')"); HF_OK = True
elif not hf: report.append("Hugging Face token : MISSING"); problems.append("HF_TOKEN is missing: add a Secret named HF_TOKEN (a token of the account that was approved for the dataset), or set DATA_SOURCE = 'files'")
else:
    try:
        from huggingface_hub import HfApi; api = HfApi(); me = api.whoami(token=hf); api.auth_check(HF_DATASET, repo_type="dataset", token=hf); HF_OK = True; os.environ["HF_TOKEN"] = hf
        report.append(f"Hugging Face token : OK (user '{me.get('name')}', access to {HF_DATASET} confirmed, {len(hf)} characters)")
    except Exception as e:
        code = getattr(getattr(e, "response", None), "status_code", None)
        why = "Hugging Face does not recognise it (wrong, expired or revoked: create a new classic 'Read' token and update the Secret)" if code == 401 else "valid token, but this account has no approved access to the gated dataset" if code == 403 else f"{type(e).__name__}: {str(e)[:100]}"
        report.append(f"Hugging Face token : REFUSED (HTTP {code}): {why}"); problems.append(f"HF_TOKEN does not work: {why}")
ku, kk = get_secret("KAGGLE_USERNAME"), get_secret("KAGGLE_KEY"); KAGGLE_OK = False
if not (ku and kk): report.append("Kaggle credentials  : not provided (the backup upload will be a dry run)"); (problems.append("KAGGLE_USERNAME / KAGGLE_KEY are missing") if REQUIRE_KAGGLE else None)
elif not shutil.which("kaggle"): report.append("Kaggle credentials  : cannot be verified (the kaggle command is not installed)"); (problems.append("the kaggle command is not installed") if REQUIRE_KAGGLE else None)
else:
    try:
        r = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True, timeout=120, env={**os.environ, "KAGGLE_USERNAME": ku, "KAGGLE_KEY": kk})
        if r.returncode == 0: KAGGLE_OK = True; os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = ku, kk; report.append(f"Kaggle credentials  : OK (user '{ku}')")
        else: report.append(f"Kaggle credentials  : REFUSED ({(r.stdout + r.stderr).strip()[:100]})"); (problems.append("the Kaggle username/key were refused (use the LEGACY API key; Internet ON?)") if REQUIRE_KAGGLE else None)
    except Exception as e: report.append(f"Kaggle credentials  : not verified ({type(e).__name__})")
print("\n".join(report))
blocking = [p for p in problems if (p.startswith("HF_TOKEN") and REQUIRE_HF) or (not p.startswith("HF_TOKEN") and REQUIRE_KAGGLE)]
if blocking: raise RuntimeError("STOPPING EARLY (before any long step):\n  - " + "\n  - ".join(blocking) + "\nFix: Add-ons > Secrets > add the missing secret(s), switch 'Attached' ON for this notebook, and run again.")
print("\nCredentials ready.")


In [ ]:
# ================= CELL 4 — LOAD AND INSPECT THE PHASE 7 DATA  [RUNS NOW; CPU] =================
# Six files, each split with ITS OWN candidate pool (train.json + train_candidates.json, dev.json + dev_candidates.json, test.json + test_candidates.json). The schema is DETECTED, not assumed: candidate files may be a dict id -> text or a list of records;
# text may be a string or a list of sentences; labels may be strings or numbers (matched to the pool's ids, so zero padding is preserved). Test labels, if a test file carries any, are IGNORED.
FILES = {"train": ("train.json", "train_candidates.json"), "dev": ("dev.json", "dev_candidates.json"), "test": ("test.json", "test_candidates.json")}
def _find(name, roots):
    hits = [p for r in roots for p in glob.glob(f"{r}/**/*", recursive=True) if os.path.basename(p).lower() == name.lower() and "phase" in p.lower() and "7" in p]
    return min(hits, key=len) if hits else None
roots = [r for r in ([LOCAL_DATA_DIR] if LOCAL_DATA_DIR else []) + ["/kaggle/input", str(WORK / "hf")] if r]
if DATA_SOURCE == "hub_files" and not all(_find(f, roots) for pair in FILES.values() for f in pair):
    from huggingface_hub import snapshot_download
    snapshot_download(HF_DATASET, repo_type="dataset", allow_patterns=[f"{DATA_SUBDIR}/*", f"{DATA_SUBDIR}/**"], token=os.environ.get("HF_TOKEN") or get_secret("HF_TOKEN"), local_dir=str(WORK / "hf")); roots.append(str(WORK / "hf"))
PATHS = {s: tuple(_find(f, roots) for f in pair) for s, pair in FILES.items()}; missing = [f for s, pair in FILES.items() for f, p in zip(pair, PATHS[s]) if not p]
assert not missing, f"Phase 7 files not found: {missing} (searched {roots}); attach them or check DATA_SUBDIR / LOCAL_DATA_DIR"
def _load(p):
    t = open(p, encoding="utf-8").read().strip()
    try: return json.loads(t)
    except json.JSONDecodeError: return [json.loads(l) for l in t.splitlines() if l.strip()]
def _text(v): return v if isinstance(v, str) else " ".join(_text(x) for x in v) if isinstance(v, (list, tuple)) else " ".join(_text(x) for x in v.values()) if isinstance(v, dict) else str(v)
ID_KEYS = ("id", "query_id", "case_id", "candidate_id", "doc_id", "qid", "cid"); TEXT_KEYS = ("text", "sentences", "document", "content", "body", "judgment", "doc")
LAB_KEYS = ("relevant_candidates", "relevant_candidate_ids", "relevant", "relevant_ids", "labels", "label", "positives", "gold", "citations", "cited", "relevant_cases")
def _records(obj):
    if isinstance(obj, dict):
        for k in ("data", "queries", "candidates", "records", "items"):
            if isinstance(obj.get(k), list): return obj[k], None
        vals = list(obj.values())[:20]
        if vals and all(isinstance(v, (str, list)) for v in vals): return [{"id": k, "text": v} for k, v in obj.items()], "dict id -> text"
        if vals and all(isinstance(v, dict) for v in vals): return [{"id": k, **v} for k, v in obj.items()], "dict id -> record"
        raise ValueError("unrecognised json object layout")
    return obj, None
def parse(obj, with_labels):
    recs, how = _records(obj); r0 = recs[0]; ik = next(k for k in ID_KEYS if k in r0); tk = next(k for k in TEXT_KEYS if k in r0); lk = next((k for k in LAB_KEYS if k in r0), None) if with_labels else None
    return [r[ik] for r in recs], [_text(r[tk]) for r in recs], ([r.get(lk) or [] for r in recs] if lk else None), dict(id=ik, text=tk, labels=lk, layout=how or "list of records")
DATA, SCHEMA = {}, {}
for s, (qp, cp) in PATHS.items():
    q_raw, q_text, q_lab, qs = parse(_load(qp), with_labels=(s != "test")); c_raw, c_text, _, cs = parse(_load(cp), with_labels=False)
    if s == "test" and any(k in _records(_load(qp))[0][0] for k in LAB_KEYS): print("[test] NOTE: test.json carries a relevance field. Test labels are private, so it is IGNORED and never read.")
    c_ids = [str(x) for x in c_raw]; by_int = {}
    for c in c_ids:
        if c.isdigit(): by_int.setdefault(int(c), c)
    def norm(lbl):
        out = []
        for x in (lbl if isinstance(lbl, (list, tuple, set)) else [lbl]):
            x = str(x)
            if x in pos_c: out.append(x)
            elif x.isdigit() and int(x) in by_int: out.append(by_int[int(x)])
            else: out.append(None)
        return out
    pos_c = {c: i for i, c in enumerate(c_ids)}; gold = None; lost = 0
    if q_lab is not None:
        gold = []
        for l in q_lab: n_ = norm(l); lost += sum(x is None for x in n_); gold.append({pos_c[x] for x in n_ if x is not None})
    DATA[s] = dict(q_ids=[str(x) for x in q_raw], q_raw=q_raw, q_text=q_text, gold=gold, c_ids=c_ids, c_raw=c_raw, c_text=c_text, c_pos=pos_c, lost_labels=lost); SCHEMA[s] = dict(queries=qs, candidates=cs)
print("SCHEMA (detected):", json.dumps(SCHEMA, indent=None)[:600])
rows = []
for s, D in DATA.items():
    wl_q, wl_c = [len(t.split()) for t in D["q_text"]], [len(t.split()) for t in D["c_text"]]; self_in = sum(q in D["c_pos"] for q in D["q_ids"])
    g = [len(x) for x in D["gold"]] if D["gold"] else []
    rows.append(dict(split=s, queries=len(D["q_ids"]), candidates=len(D["c_ids"]), query_words_p50=int(np.median(wl_q)), cand_words_p50=int(np.median(wl_c)), cand_words_p90=int(np.percentile(wl_c, 90)), gold_mean=round(float(np.mean(g)), 2) if g else None,
                     gold_p50=float(np.median(g)) if g else None, gold_max=max(g) if g else None, no_gold=sum(x == 0 for x in g) if g else None, labels_not_in_pool=D["lost_labels"], queries_inside_own_pool=self_in, dup_cand_ids=len(D["c_ids"]) - len(set(D["c_ids"]))))
print(pd.DataFrame(rows).to_string(index=False))
pools = {s: set(D["c_ids"]) for s, D in DATA.items()}; print("candidate-pool overlap between splits:", {f"{a}&{b}": len(pools[a] & pools[b]) for a, b in (("train", "dev"), ("train", "test"), ("dev", "test"))}, "| every split is retrieved ONLY from its own pool")
GOLD_MEAN = float(np.mean([len(x) for x in DATA["dev"]["gold"]])) if DATA["dev"]["gold"] else None; print(f"dev: a query has {GOLD_MEAN:.2f} relevant candidates on average; the evaluator compares the top {TOP_K} with the WHOLE gold set (micro-F1@{TOP_K})")


In [ ]:
# ================= CELL 5 — PHASE 1 TOKENIZER AND CHUNKS  [RUNS NOW; CPU, a few minutes] =================
# Every document keeps its token ids; chunks (512 tokens, spread EVENLY over the document) are cut on demand, so the number of chunks can be changed without tokenising again (MAX_CHUNKS for training / default scoring, MAX_CHUNKS_BIG for the chunk sweep).
from tokenizers import Tokenizer
tpaths = [TOKENIZER_PATH] if TOKENIZER_PATH else sorted(glob.glob("/kaggle/input/**/legal_bpe_32k_mask.json", recursive=True)) or sorted(glob.glob("/kaggle/input/**/legal_bpe_32k*.json", recursive=True))
assert tpaths, "the Phase 1 tokenizer (legal_bpe_32k_mask.json) was not found: attach a Phase 1 dataset"
tok = Tokenizer.from_file(tpaths[0]); VOCAB = tok.get_vocab_size(); PAD = tok.token_to_id("[PAD]") or 0; BOS, EOS = tok.token_to_id("[BOS]"), tok.token_to_id("[EOS]")
CHUNK_TOK, MAX_CHUNKS, MAX_CHUNKS_BIG = 512, 12, 24
if SELFTEST: CHUNK_TOK, MAX_CHUNKS, MAX_CHUNKS_BIG = 48, 3, 6
def chunk_doc(ids, max_chunks=None):
    body = CHUNK_TOK - 2; n = max(1, math.ceil(len(ids) / body)); m = max_chunks or MAX_CHUNKS; pick = sorted(set(np.linspace(0, n - 1, min(n, m)).round().astype(int).tolist())); out = []
    for k in pick:
        seg = list(ids[k * body:(k + 1) * body])
        if seg or k == 0: out.append(np.asarray(([BOS] if BOS is not None else []) + seg + ([EOS] if EOS is not None else []), dtype=np.int32))
    return out
t0 = time.time(); TOK_LEN = {}
for s, D in DATA.items():
    for side in ("q", "c"):
        encs = tok.encode_batch(D[f"{side}_text"], add_special_tokens=False); D[f"{side}_tok"] = [np.asarray(e.ids, dtype=np.int32) for e in encs]; D[f"{side}_chunks"] = [chunk_doc(i) for i in D[f"{side}_tok"]]; TOK_LEN[(s, side)] = [len(i) for i in D[f"{side}_tok"]]; del encs
_CH_CACHE = {}
def chunks_for(s, side, m=None):
    m = m or MAX_CHUNKS
    if m == MAX_CHUNKS: return DATA[s][f"{side}_chunks"]
    if (s, side, m) not in _CH_CACHE: _CH_CACHE[(s, side, m)] = [chunk_doc(i, m) for i in DATA[s][f"{side}_tok"]]
    return _CH_CACHE[(s, side, m)]
print(f"tokenised every document in {time.time() - t0:.0f}s | chunk {CHUNK_TOK} tokens, {MAX_CHUNKS} chunks by default ({MAX_CHUNKS_BIG} in the sweep), vocabulary {VOCAB}")
print(pd.DataFrame([dict(split=s, side={"q": "queries", "c": "candidates"}[side], tokens_p50=int(np.median(v)), tokens_p90=int(np.percentile(v, 90)), longer_than_default_chunks=f"{np.mean(np.asarray(v) > MAX_CHUNKS * (CHUNK_TOK - 2)):.0%}", longer_than_big=f"{np.mean(np.asarray(v) > MAX_CHUNKS_BIG * (CHUNK_TOK - 2)):.0%}") for (s, side), v in TOK_LEN.items()]).to_string(index=False))


In [ ]:
# ================= CELL 8 — LOAD THE PHASE 1 MODELS  [NEEDS the Phase 1 checkpoints; set CKPT_PATHS in the config cell or attach the datasets]. The old 3,000-step GPT-2 is never used. =================
ARCH_LOADERS_SRC = r'''"""arch_loaders.py - load any of the four pre-trained architectures as a backbone for classification / tagging.
   load_backbone(path) -> (backbone, hidden_size, causal, max_len, family)
   backbone(input_ids, attention_mask) -> object with .last_hidden_state of shape [B, T, hidden_size]"""
import torch, torch.nn as nn
from types import SimpleNamespace
from transformers import GPT2Config, GPT2LMHeadModel, ModernBertConfig, ModernBertForMaskedLM, T5Config, T5ForConditionalGeneration

class BiLSTMLM(nn.Module):
    def __init__(self, vocab, emb=512, hidden=1024, layers=3):
        super().__init__()
        self.vocab, self.emb_dim, self.hidden, self.layers = vocab, emb, hidden, layers
        self.emb = nn.Embedding(vocab, emb); self.fwd = nn.LSTM(emb, hidden, layers, batch_first=True); self.bwd = nn.LSTM(emb, hidden, layers, batch_first=True)
        self.norm_f, self.norm_b = nn.LayerNorm(hidden), nn.LayerNorm(hidden)
        self.proj_f, self.proj_b = nn.Linear(hidden, emb), nn.Linear(hidden, emb); self.out_bias = nn.Parameter(torch.zeros(vocab))
    def hidden_states(self, input_ids, attention_mask=None):
        x = input_ids; B, T = x.shape
        lengths = attention_mask.sum(1) if attention_mask is not None else torch.full((B,), T, device=x.device)
        rev = (lengths[:, None] - 1 - torch.arange(T, device=x.device)[None, :]).clamp(min=0)
        hf, _ = self.fwd(self.emb(x)); hb, _ = self.bwd(self.emb(x.gather(1, rev)))
        hb = hb.gather(1, rev[:, :, None].expand(-1, -1, hb.shape[-1]))
        return torch.cat([hf, hb], dim=-1)

class Backbone(nn.Module):
    """one calling convention for every family (matches what the Phase 3 SeqClassifier expects)"""
    def __init__(self, family, inner, embed):
        super().__init__(); self.family, self.inner, self._embed = family, inner, embed
    def get_input_embeddings(self): return self._embed
    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None, use_cache=False, **kw):
        f = self.family
        if f == "gpt2": h = self.inner(input_ids=input_ids, attention_mask=attention_mask, inputs_embeds=inputs_embeds, use_cache=False).last_hidden_state
        else:
            assert inputs_embeds is None, f"prompt tuning (inputs_embeds) is only wired for the gpt2 family, not {f}"
            if f in ("modernbert", "t5"): h = self.inner(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
            elif f == "bilstm": h = self.inner.hidden_states(input_ids, attention_mask)
            else: raise ValueError(f)
        return SimpleNamespace(last_hidden_state=h)

CAUSAL = {"gpt2": True, "modernbert": False, "t5": False, "bilstm": False}
# modules to adapt with LoRA / LoRMA / IA3 (None = full fine-tuning only). ModernBERT's q,k,v are ONE fused linear, 'Wqkv' (rows: q | k | v).
ADAPTER_TARGETS = {"gpt2": "attn.c_attn (fused Conv1D: q|k|v)", "modernbert": "attn.Wqkv (fused Linear: q|k|v)", "t5": "SelfAttention q, k, v, o (Linear)", "bilstm": None}

def build_full(mc):
    f, V, d = mc["family"], mc["vocab_size"], mc["hidden_size"]
    if f == "gpt2":
        return GPT2LMHeadModel(GPT2Config(vocab_size=V, n_positions=mc["context_length"], n_ctx=mc["context_length"], n_embd=d, n_layer=mc["num_layers"], n_head=mc["num_heads"], n_inner=mc.get("n_inner", 4 * d),
                                          resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0))
    if f == "modernbert":
        return ModernBertForMaskedLM(ModernBertConfig(vocab_size=V, max_position_embeddings=mc["context_length"], hidden_size=d, intermediate_size=mc["intermediate_size"], num_hidden_layers=mc["num_layers"],
                                                      num_attention_heads=mc["num_heads"], local_attention=mc["local_attention"], global_attn_every_n_layers=mc["global_attn_every_n_layers"], pad_token_id=0,
                                                      attention_dropout=0.0, mlp_dropout=0.0, embedding_dropout=0.0, classifier_pooling="mean", sparse_prediction=True, sparse_pred_ignore_index=-100, reference_compile=False))
    if f == "t5":
        return T5ForConditionalGeneration(T5Config(vocab_size=V, d_model=d, d_kv=mc["d_kv"], d_ff=mc["d_ff"], num_layers=mc["num_layers"], num_decoder_layers=mc["num_decoder_layers"], num_heads=mc["num_heads"],
                                                   feed_forward_proj="gated-gelu", dropout_rate=0.0, tie_word_embeddings=True))
    if f == "bilstm": return BiLSTMLM(V, emb=mc["emb"], hidden=mc["lstm_hidden"], layers=mc["num_layers"])
    raise ValueError(f)

def load_backbone(path, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False); mc = ck["model_config"]; full = build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True)
    f = mc["family"]
    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte
    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings
    elif f == "t5": inner, embed = full.encoder, full.shared
    elif f == "bilstm": inner, embed = full, full.emb
    else: raise ValueError(f)
    return Backbone(f, inner, embed).to(device), mc["hidden_size"], CAUSAL[f], mc["context_length"], f
'''
al_paths = sorted(glob.glob("/kaggle/input/**/arch_loaders.py", recursive=True)) or sorted(glob.glob("/kaggle/working/**/arch_loaders.py", recursive=True))
if not al_paths:                                                                  # self-contained: the loader module is embedded in this notebook
    (WORK / "arch_loaders.py").write_text(ARCH_LOADERS_SRC); al_paths = [str(WORK / "arch_loaders.py")]; print("arch_loaders.py not found among the inputs: using the copy embedded in this notebook")
sys.path.insert(0, str(Path(al_paths[0]).parent)); import arch_loaders as AL

MODEL_PATHS = {}                                    # filled automatically below; you can also set entries by hand, e.g. MODEL_PATHS["bilstm"] = "/kaggle/input/.../final_model.pt"
def discover_models():
    found = {}
    for p in sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True)):
        if os.path.getsize(p) < 50e6: continue
        base = os.path.basename(p)
        if base != "final_model.pt": continue                # only the NEW Phase 1 models: the old 3,000-step checkpoint is deliberately never used
        try: ck = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
        except Exception: continue
        if not isinstance(ck, dict) or "model_config" not in ck or "model_state_dict" not in ck: continue
        mc = ck["model_config"]; fam = mc.get("family")
        if fam is None: continue                              # a checkpoint without a family entry is an old-format model: skipped
        key = "gpt2_new" if fam == "gpt2" else fam
        if key not in found: found[key] = p
    return found
MODEL_PATHS.update({k: v for k, v in discover_models().items() if k not in MODEL_PATHS}); MODEL_PATHS.update({k: v for k, v in CKPT_PATHS.items() if v})        # CKPT_PATHS from the config cell wins
print("Phase 1 models found:", {k: v for k, v in MODEL_PATHS.items()}, "\n  missing (pre-train them first):", [k for k in ("gpt2_new", "modernbert", "t5", "bilstm") if k not in MODEL_PATHS])

def load_backbone_any(path):
    """like AL.load_backbone, but also accepts the OLD Phase 1 checkpoint (its model_config has no 'family' entry)"""
    ck = torch.load(path, map_location="cpu", weights_only=False); mc = dict(ck["model_config"]); mc.setdefault("family", "gpt2")
    full = AL.build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True); f = mc["family"]
    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte
    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings
    elif f == "t5": inner, embed = full.encoder, full.shared
    elif f == "bilstm": inner, embed = full, full.emb
    else: raise ValueError(f)
    del ck; return AL.Backbone(f, inner, embed), mc["hidden_size"], AL.CAUSAL[f], mc["context_length"], f

def set_backbone_dropout(backbone, p):
    """turn on dropout inside the backbone for fine-tuning (works where the model has nn.Dropout modules, e.g. GPT-2 and T5; a no-op for the others). Returns how many modules were changed."""
    n = 0
    for mod in backbone.modules():
        if isinstance(mod, nn.Dropout): mod.p = p; n += 1
    return n

def truncate_layers(bb, n):
    """keep only the first n transformer layers (initialises a smaller student from the same Phase 1 weights); implemented for the GPT-2 family"""
    assert bb.family == "gpt2", "layer truncation is implemented for the GPT-2 family only"
    bb.inner.h = nn.ModuleList(list(bb.inner.h)[:n]); bb.inner.config.n_layer = n; return bb

def refresh_models(verbose=True):
    """re-scan /kaggle/input for Phase 1 models (for example after you attached another dataset to the running session)"""
    new = {k: v for k, v in discover_models().items() if k not in MODEL_PATHS}; MODEL_PATHS.update(new)
    if verbose and new: print("newly found Phase 1 models:", new)
    return sorted(MODEL_PATHS)

class MLClassifier(nn.Module):
    """Phase 1 backbone + a multi-label head. One window of <= L tokens in, one logit per statute out. Causal models (GPT-2) pool the last token, the others the mean of the tokens."""
    def __init__(self, backbone, hidden, causal, n_labels, dropout=0.1, pool="last"):
        super().__init__(); self.backbone, self.hidden, self.causal, self.n_labels, self.pool = backbone, hidden, causal, n_labels, pool
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, n_labels))
    def forward(self, ids, mask):
        T = int(mask.sum(1).max()); ids, mask = ids[:, :T], mask[:, :T]
        h = self.backbone(input_ids=ids, attention_mask=mask).last_hidden_state; mm = mask.unsqueeze(-1).to(h.dtype); mean = (h * mm).sum(1) / mm.sum(1).clamp(min=1)
        if self.causal:
            last = h[torch.arange(h.shape[0], device=h.device), mask.sum(1) - 1]; pooled = last if self.pool == "last" else 0.5 * (last + mean)
        else: pooled = mean
        return self.head(pooled)

# ---- windows over the stored tokens of a case (the whole case if <= 8,192 tokens, else its first 4,096 + last 4,096) ----
def window_spans(n, L, W, order):
    """up to W (start, end) spans of <= L tokens. order: 'head' (from the start), 'tail' (from the end), 'both' (head and tail windows interleaved, skipping a window already covered)"""
    nb = max(1, -(-n // L)); head = [(k * L, min(n, (k + 1) * L)) for k in range(nb)]; tail = [(max(0, n - (k + 1) * L), n - k * L) for k in range(nb)]
    if order == "head": sp = head
    elif order == "tail": sp = tail
    else:
        sp = []
        for a, b in zip(head, tail):
            for s_ in (a, b):
                if not any(s_[0] >= c[0] and s_[1] <= c[1] for c in sp): sp.append(s_)
    return sp[:W]
def build_batch(toks, idx, spans, L):
    ids = np.full((len(idx), L), PAD, dtype=np.int64); mask = np.zeros((len(idx), L), dtype=np.int64)
    for b, (i, (s, e)) in enumerate(zip(idx, spans)):
        seg = toks[i][s:e]; ids[b, :len(seg)] = seg; mask[b, :len(seg)] = 1
    return torch.from_numpy(ids), torch.from_numpy(mask)


In [ ]:
# ================= CELL 9 — ADAPTATION TECHNIQUES  [pure code]: full, partial (layer-wise freezing), LoRA, rsLoRA, LoRMA+, LoRMA-pi, IA3 =================
from transformers.pytorch_utils import Conv1D
class LoRADelta(nn.Module):                # additive:  dW x = (alpha/r) B A x   (rsLoRA: alpha/sqrt(r))
    def __init__(self, nx, nf, r, alpha, drop, rs=False):
        super().__init__()
        self.A = nn.Parameter(torch.empty(nx, r).uniform_(-1 / math.sqrt(nx), 1 / math.sqrt(nx))); self.B = nn.Parameter(torch.zeros(r, nf))
        self.scale = alpha / (math.sqrt(r) if rs else r); self.drop = nn.Dropout(drop)
    def forward(self, x): return (self.drop(x) @ self.A) @ self.B * self.scale
class LoRMAPlus(nn.Module):                # multiplicative, additive rank inflation: h' = (I + (alpha/r) B A) h ; B = 0 at init  (= W0 x + (alpha/r) B A (W0 x))
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d))); self.B = nn.Parameter(torch.zeros(d, r)); self.scale = alpha / r
    def forward(self, h): return h + ((h @ self.A.t()) @ self.B.t()) * self.scale
class LoRMAPi(nn.Module):                  # multiplicative, permutation-based rank inflation: h' = I_pi(BA) h, I_pi(BA) = I at init
    def __init__(self, d, r):
        super().__init__()
        B = torch.empty(d, r).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d)); B[:, 0] = 1.0; A = torch.zeros(r, d); A[0, 0] = 1.0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A); i = torch.arange(d); self.register_buffer("idx", (i[None, :] - i[:, None]) % d)
    def matrix(self): return (self.B @ self.A).gather(1, self.idx)
    def forward(self, h): return h @ self.matrix().t().to(h.dtype)
class _Wrap:
    """expose the wrapped layer's weight / bias: some Hugging Face modules read them directly (T5's feed-forward dtype check does)"""
    @property
    def weight(self): return self.base.weight
    @property
    def bias(self): return self.base.bias
def proj_dims(m):
    return (m.in_features, m.out_features) if isinstance(m, nn.Linear) else tuple(m.weight.shape)      # HF Conv1D weight is [in, out]
class FusedQV(_Wrap, nn.Module):                  # wraps a FUSED q|k|v projection (GPT-2 Conv1D c_attn, ModernBERT Linear Wqkv); LoRA / LoRMA adapt q and v, IA3 rescales k and v
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); nx, nf = proj_dims(base); d = nf // 3; self.base, self.kind, self.d = base, kind, d
        if kind == "lora": self.q, self.v = LoRADelta(nx, d, r, alpha, drop, rs), LoRADelta(nx, d, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.q, self.v = LoRMAPlus(d, r, alpha), LoRMAPlus(d, r, alpha)
        elif kind == "lorma_pi": self.q, self.v = LoRMAPi(d, r), LoRMAPi(d, r)
        elif kind == "ia3": self.lk, self.lv = nn.Parameter(torch.ones(d)), nn.Parameter(torch.ones(d))
        else: raise ValueError(kind)
    def forward(self, x):
        out = self.base(x); d = self.d; q, k, v = out[..., :d], out[..., d:2 * d], out[..., 2 * d:]
        if self.kind == "lora": q, v = q + self.q(x), v + self.v(x)
        elif self.kind.startswith("lorma"): q, v = self.q(q), self.v(v)
        else: k, v = k * self.lk, v * self.lv
        return torch.cat([q, k, v], -1)
class IA3Input(_Wrap, nn.Module):                 # IA3 on the FFN: rescale the activation entering the down-projection
    def __init__(self, base): super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(proj_dims(base)[0]))
    def forward(self, x): return self.base(x * self.l)
class LinAdapt(_Wrap, nn.Module):                 # the same adapters for separate nn.Linear projections (T5's q, k, v)
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); self.base, self.kind = base, kind; di, do = base.in_features, base.out_features
        if kind == "lora": self.op = LoRADelta(di, do, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.op = LoRMAPlus(do, r, alpha)
        elif kind == "lorma_pi": self.op = LoRMAPi(do, r)
        elif kind == "ia3": self.l = nn.Parameter(torch.ones(do))
    def forward(self, x):
        y = self.base(x)
        if self.kind == "lora": return y + self.op(x)
        if self.kind == "ia3": return y * self.l
        return self.op(y)
def wrap_linear_targets(root, kind, names, **kw):
    n = 0
    for _, parent in list(root.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in names: setattr(parent, cname, LinAdapt(child, kind, **kw)); n += 1
    assert n > 0, f"no nn.Linear named {names} found in the backbone"
def freeze_lower(model, frac=0.5):
    """partial / layer-wise fine-tuning: freeze the embeddings and the lowest `frac` of the layers, train the rest"""
    inner, fam = model.backbone.inner, model.backbone.family
    for p in inner.parameters(): p.requires_grad = True
    if fam == "bilstm":
        n = inner.layers; k = int(n * frac)
        for p in inner.emb.parameters(): p.requires_grad = False
        for name, p in list(inner.fwd.named_parameters()) + list(inner.bwd.named_parameters()):
            if int(name.split("_l")[-1]) < k: p.requires_grad = False
        return
    if fam == "gpt2": layers, pre = list(inner.h), [inner.wte, inner.wpe]
    elif fam == "modernbert": layers, pre = list(inner.layers), [inner.embeddings]
    elif fam == "t5": layers, pre = list(inner.block), [inner.embed_tokens]
    else: raise ValueError(fam)
    k = int(len(layers) * frac)
    for mod in pre:
        for p in mod.parameters(): p.requires_grad = False
    for lay in layers[:k]:
        for p in lay.parameters(): p.requires_grad = False
METHODS = ("full", "partial", "lora", "rslora", "lorma_plus", "lorma_pi", "ia3")
ADAPTER_LR = {"lora": 5e-4, "rslora": 5e-4, "lorma_plus": 5e-4, "lorma_pi": 2e-4, "ia3": 3e-3}        # starting points; use lr_pilot(..., method=...) to check them
def apply_method(model, method, hp=None):
    assert method in METHODS, method; hp = hp or {}
    if method == "full":
        for p in model.backbone.parameters(): p.requires_grad = True
        return model
    if method == "partial": freeze_lower(model, hp.get("frac", 0.5)); return model
    fam, inner = model.backbone.family, model.backbone.inner
    for p in model.backbone.parameters(): p.requires_grad = False
    kind = {"lora": "lora", "rslora": "lora", "lorma_plus": "lorma_plus", "lorma_pi": "lorma_pi", "ia3": "ia3"}[method]; kw = dict(r=hp.get("r", 16), alpha=hp.get("alpha", 32), drop=hp.get("adapter_dropout", 0.05), rs=(method == "rslora"))
    if fam == "gpt2":
        for blk in inner.h:
            assert isinstance(blk.attn.c_attn, Conv1D), "unexpected c_attn type in this transformers version"; blk.attn.c_attn = FusedQV(blk.attn.c_attn, kind, **kw)
            if kind == "ia3": blk.mlp.c_proj = IA3Input(blk.mlp.c_proj)
    elif fam == "modernbert":
        for lay in inner.layers:
            assert isinstance(lay.attn.Wqkv, nn.Linear); lay.attn.Wqkv = FusedQV(lay.attn.Wqkv, kind, **kw)
            if kind == "ia3": lay.mlp.Wo = IA3Input(lay.mlp.Wo)
    elif fam == "t5":
        wrap_linear_targets(inner, kind, ("k", "v") if kind == "ia3" else ("q", "v"), **({} if kind == "ia3" else kw))
        if kind == "ia3":
            for _, parent in list(inner.named_modules()):
                if "DenseGatedActDense" in type(parent).__name__ or "DenseReluDense" in type(parent).__name__: parent.wo = IA3Input(parent.wo)
    else: raise NotImplementedError(f"'{method}' needs attention/linear layers: the {fam} model supports only 'full' and 'partial'")
    return model


In [ ]:
# ================= CELL 8 — METRICS, RANKING AND THE SUBMISSION FILE  [pure code] =================
# micro-P@7 = sum of hits / (7 x queries); micro-R@7 = sum of hits / sum of gold sizes (the WHOLE gold set, neither truncated nor padded); micro-F1@7 = their harmonic mean.
def rank_top(scores, D, k=TOP_K):
    """scores [n_queries, n_candidates] -> for every query the indices of its top-k candidates (the query itself is never returned if it sits in its own pool)"""
    S = np.array(scores, dtype=np.float32, copy=True)
    for qi, q in enumerate(D["q_ids"]):
        if q in D["c_pos"]: S[qi, D["c_pos"][q]] = -np.inf
    return np.argsort(-S, axis=1)[:, :k]
def micro_at_k(top, gold, k=TOP_K):
    hits = np.array([len(set(t[:k].tolist()) & g) for t, g in zip(top, gold)], dtype=float); gs = np.array([len(g) for g in gold], dtype=float)
    p = hits.sum() / (k * len(gold)); r = hits.sum() / max(1.0, gs.sum()); f = 0.0 if p + r == 0 else 2 * p * r / (p + r); return dict(f1=f, p=p, r=r, hits=hits, gsize=gs)
def boot_diff(m_a, m_b, n=1000, seed=0, k=TOP_K):
    """paired bootstrap over QUERIES of the micro-F1@k difference a - b: (mean, 2.5%, 97.5%)"""
    r = np.random.default_rng(seed); ha, hb, gs = m_a["hits"], m_b["hits"], m_a["gsize"]; nq = len(gs); out = []
    for _ in range(n):
        i = r.integers(0, nq, nq); f = lambda h: (lambda p, rr: 0.0 if p + rr == 0 else 2 * p * rr / (p + rr))(h[i].sum() / (k * nq), h[i].sum() / max(1.0, gs[i].sum())); out.append(f(ha) - f(hb))
    out = np.array(out); return float(out.mean()), float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))
def write_prediction(path, top, D):
    sub = {D["q_raw"][qi] if isinstance(D["q_raw"][qi], str) else str(D["q_raw"][qi]): [D["c_raw"][ci] if isinstance(D["c_raw"][ci], str) else str(D["c_raw"][ci]) for ci in top[qi][:TOP_K]] for qi in range(len(D["q_ids"]))}
    json.dump(sub, open(path, "w", encoding="utf-8"), ensure_ascii=False); return sub
def validate_prediction(path, D):
    sub = json.load(open(path, encoding="utf-8")); pool = set(map(str, D["c_raw"])); qs = list(map(str, D["q_raw"]))
    assert isinstance(sub, dict), "prediction.json must be a JSON object: query id -> list of candidate ids"
    assert set(sub) == set(qs) and len(sub) == len(qs), f"query ids differ from the test file ({len(set(sub) ^ set(qs))} differ)"
    for q, lst in sub.items():
        assert isinstance(lst, list) and len(lst) == TOP_K, f"{q}: {len(lst)} candidates instead of {TOP_K}"
        assert len(set(lst)) == TOP_K, f"{q}: duplicate candidates"; assert all(c in pool for c in lst), f"{q}: a candidate outside the TEST pool"; assert q not in lst, f"{q}: the query itself was returned"
    return True
def zip_prediction(path, zpath):
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z: z.write(path, arcname="prediction.json")          # the file at the ROOT of the archive, no folder


In [ ]:
# ================= CELL 12 — THE ROLLING FINAL FILE  [pure code] =================
# After EVERY finished run the best RULE-SAFE system so far (by dev micro-F1@7) writes prediction.json + prediction.zip, so a valid file always exists even if the session stops early.
CKPT = WORK / "checkpoints"; CKPT.mkdir(parents=True, exist_ok=True)
def safe(label, fn):
    try: return fn()
    except Exception as e:
        import traceback; print(f"!! {label} FAILED ({type(e).__name__}: {str(e)[:200]}); the notebook carries on"); traceback.print_exc(limit=3); return None
PRED_PATH = WORK / "prediction_SELFTEST.json" if SELFTEST else Path("/kaggle/working/prediction.json"); ZIP_PATH = PRED_PATH.with_suffix(".zip")
def runs_table():
    rows = [json.load(open(p)) for p in (WORK / "runs").glob("*/result.json")]; return pd.DataFrame(rows) if rows else pd.DataFrame()
def safe_rule(r): return r.get("technique") != "bm25" and not str(r.get("name", "")).startswith("H_")          # rule-safe = only Phase 1 models (and their ensembles)
def allowed(r):
    a = r.get("adopted", True); ok = safe_rule(r) if RULE_MODE == "strict" else r.get("technique") != "bm25"
    return ok and not (a is False or str(a) == "False")                                                  # a stage that did not pass its adoption rule is kept in the table but never becomes prediction.json                       # "hybrid_ok" also allows Phase 1 + BM25 hybrids; pure BM25 is NEVER a submission
def update_final(verbose=True, names=None):
    T = runs_table()
    if not len(T): return None
    T = T[[allowed(r) for r in T.to_dict("records")]]
    if names is not None: T = T[T.name.isin(names)]
    if not len(T): return None
    best = T.sort_values("dev_f1", ascending=False).iloc[0]; St = np.load(WORK / "runs" / best["name"] / "test_scores.npy").astype(np.float32)
    write_prediction(PRED_PATH, rank_top(St, DATA["test"]), DATA["test"]); validate_prediction(PRED_PATH, DATA["test"]); zip_prediction(PRED_PATH, ZIP_PATH); json.dump(dict(system=best["name"], dev_f1=float(best["dev_f1"])), open(WORK / "final_choice.json", "w"))
    if verbose: print(f"prediction.json <- {best['name']} (dev micro-F1@7 {best['dev_f1']:.4f}) | validated: {len(DATA['test']['q_ids'])} test queries x {TOP_K} unique candidates from the TEST pool | zip: {ZIP_PATH.name}")
    return best["name"]


In [ ]:
# ================= CELL 10 — THE RETRIEVAL ENCODER, ENCODING AND SCORING  [pure code] =================
# A BI-ENCODER on the Phase 1 ModernBERT: every 512-token chunk -> the mean of its token states -> a unit vector. Two ways to score a query against a candidate (both measured on dev; the better one is used per run):
#   'mean'   cosine of the mean chunk vectors (a document-level representation)
#   'maxsim' for every query chunk the best-matching candidate chunk, averaged over the query chunks (chunk-level, keeps WHERE the documents agree)
def truncate_modernbert(bb, n):
    bb.inner.layers = nn.ModuleList(list(bb.inner.layers)[:n]); bb.inner.config.num_hidden_layers = n; return bb
class DocEncoder(nn.Module):
    def __init__(self, bb, hidden, n_prompt=0):
        super().__init__(); self.backbone, self.hidden, self.n_prompt = bb, hidden, n_prompt
        if n_prompt:                                                             # prompt tuning: n_prompt trained vectors in front of every chunk, initialised from real token embeddings
            E = bb.inner.embeddings.tok_embeddings.weight; self.prompt = nn.Parameter(E[torch.randint(10, E.shape[0], (n_prompt,))].detach().clone())
    def forward(self, ids, mask):
        T = int(mask.sum(1).max()); ids, mask = ids[:, :T], mask[:, :T]
        if self.n_prompt:
            e = self.backbone.inner.embeddings.tok_embeddings(ids); P = self.prompt.to(e.dtype).unsqueeze(0).expand(ids.shape[0], -1, -1)
            h = self.backbone.inner(inputs_embeds=torch.cat([P, e], 1), attention_mask=torch.cat([torch.ones(ids.shape[0], self.n_prompt, dtype=mask.dtype, device=mask.device), mask], 1)).last_hidden_state[:, self.n_prompt:]
        else: h = self.backbone(input_ids=ids, attention_mask=mask).last_hidden_state
        mm = mask.unsqueeze(-1).to(h.dtype); return F.normalize(((h * mm).sum(1) / mm.sum(1).clamp(min=1)).float(), dim=-1)
def pad_chunks(chs):
    L = max(len(c) for c in chs); ids = torch.full((len(chs), L), PAD, dtype=torch.long); mask = torch.zeros((len(chs), L), dtype=torch.long)
    for i, c in enumerate(chs): ids[i, :len(c)] = torch.from_numpy(c.astype(np.int64)); mask[i, :len(c)] = 1
    return ids, mask
@torch.no_grad()
def encode_docs(model, docs, bs=None):
    """docs: list of chunk lists -> (E [n_docs, max_chunks, H] float16, M [n_docs, max_chunks] bool) on the CPU"""
    model.eval(); flat = [(d, k, c) for d, chs in enumerate(docs) for k, c in enumerate(chs)]; order = sorted(range(len(flat)), key=lambda i: len(flat[i][2])); bs = bs or (64 if device.type == "cuda" else 16)
    kmax = max(len(chs) for chs in docs); E = torch.zeros(len(docs), kmax, model.hidden, dtype=torch.float16); M = torch.zeros(len(docs), kmax, dtype=torch.bool)
    for j in range(0, len(order), bs):
        part = [flat[i] for i in order[j:j + bs]]; ids, mask = pad_chunks([p[2] for p in part])
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP): v = model(ids.to(device), mask.to(device))
        v = v.half().cpu()
        for (d, k, _), x in zip(part, v): E[d, k] = x; M[d, k] = True
    return E, M
@torch.no_grad()
def score(QE, QM, CE, CM, mode):
    if mode == "mean":
        q = F.normalize((QE.float() * QM[..., None]).sum(1) / QM.sum(1, keepdim=True).clamp(min=1), dim=-1); c = F.normalize((CE.float() * CM[..., None]).sum(1) / CM.sum(1, keepdim=True).clamp(min=1), dim=-1); return (q @ c.T).numpy()
    dev_ = device; C = CE.to(dev_).float(); Cm = CM.to(dev_); out = np.zeros((QE.shape[0], CE.shape[0]), dtype=np.float32)
    for i in range(0, QE.shape[0], 8):
        q = QE[i:i + 8].to(dev_).float(); qm = QM[i:i + 8].to(dev_); s = torch.einsum("qah,cbh->qcab", q, C).masked_fill(~Cm[None, :, None, :], -1e4).amax(-1)
        out[i:i + 8] = ((s * qm[:, None, :]).sum(-1) / qm.sum(-1, keepdim=True).clamp(min=1)).cpu().numpy()
    return out
def evaluate_model(model, name, technique, note, splits=("dev", "test"), extra=None, control=False, chunk_m=None, adopted=True):
    """encode dev + test (queries and their OWN pools), score both ways, keep the better mode on dev, save everything"""
    t0 = time.time(); enc = {s: (encode_docs(model, chunks_for(s, "q", chunk_m)), encode_docs(model, chunks_for(s, "c", chunk_m))) for s in splits}; res = {}
    for mode in ("mean", "maxsim"): S = score(*enc["dev"][0], *enc["dev"][1], mode); res[mode] = (micro_at_k(rank_top(S, DATA["dev"]), DATA["dev"]["gold"]), S)
    mode = max(res, key=lambda m: res[m][0]["f1"]); m, Sd = res[mode]; St = score(*enc["test"][0], *enc["test"][1], mode); secs = time.time() - t0
    print(f"[{name}] dev micro-F1@7 {m['f1']:.4f} (P@7 {m['p']:.4f}, R@7 {m['r']:.4f}) with '{mode}' scoring | other mode: {res['mean' if mode == 'maxsim' else 'maxsim'][0]['f1']:.4f} | encoding + scoring {secs / 60:.1f} min")
    save_scores(name, technique, note, Sd, St, m, extra=dict(mode=mode, chunks=chunk_m or MAX_CHUNKS, other_mode_f1=res["mean" if mode == "maxsim" else "maxsim"][0]["f1"], eval_minutes=secs / 60, **(extra or {})), control=control, adopted=adopted); return m
def save_scores(name, technique, note, S_dev, S_test, m, extra=None, control=False, adopted=True):
    rd = WORK / "runs" / name; rd.mkdir(parents=True, exist_ok=True); np.save(rd / "dev_scores.npy", S_dev.astype(np.float16)); np.save(rd / "test_scores.npy", S_test.astype(np.float16))
    res = dict(name=name, technique=technique, note=note, control=control, adopted=bool(adopted), dev_f1=m["f1"], dev_p=m["p"], dev_r=m["r"], dev_queries=len(DATA["dev"]["q_ids"]), **(extra or {})); json.dump(res, open(rd / "result.json", "w"), indent=1, default=str)
    led = pd.read_csv(LEDGER) if LEDGER.exists() else pd.DataFrame(); led = pd.concat([led[led.name != name] if len(led) else led, pd.DataFrame([{k: v for k, v in res.items() if not isinstance(v, (dict, list))}])]); led.to_csv(LEDGER, index=False)
    update_final(verbose=False); return res
def run_done(name): return (WORK / "runs" / name / "result.json").exists() and not RERUN


In [ ]:
# ================= CELL 13 — RESTORE  [optional; RUNS NOW] =================
# (a) an earlier output of THIS notebook (pcr_bilstm/), (b) the retriever dataset this notebook creates (flat file names + a manifest). With either attached (and RETRAIN = False) training is skipped and only the predictions are regenerated.
RUN_FILES = ("result.json", "dev_scores.npy", "test_scores.npy"); RESTORED = set()
for src in sorted({Path(p).parent for p in glob.glob("/kaggle/input/**/pcr_bilstm/ledger.csv", recursive=True)} - {WORK}):
    for sub in ("runs", "checkpoints"):
        if (src / sub).exists(): shutil.copytree(src / sub, WORK / sub, dirs_exist_ok=True)
    RESTORED.add(str(src)); print("restored an earlier output:", src)
for mf in sorted(glob.glob("/kaggle/input/**/BILSTM_RETRIEVER_MANIFEST.json", recursive=True)):
    src = Path(mf).parent; man = json.load(open(mf)); bad = []
    for name, meta in man["files"].items():
        f = src / name
        if not f.exists() or f.stat().st_size != meta["bytes"]: bad.append(name); continue
        if name.startswith("ckpt__"): dst = CKPT / name[6:]
        elif "__" in name: run, fn = name.split("__", 1); dst = WORK / "runs" / run / fn
        else: continue                                                                    # arch_loaders.py, README.md, results.json: documentation, nothing to restore
        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(f, dst)
    RESTORED.add(str(src)); print(f"restored the retriever dataset {src}" + (f" | missing or wrong size: {bad}" if bad else " | all sizes match the manifest"))
rows = [json.load(open(p)) for p in (WORK / "runs").glob("*/result.json")]
if rows: pd.DataFrame([{k: v for k, v in r.items() if not isinstance(v, (dict, list))} for r in rows]).to_csv(LEDGER, index=False)
SKIP_TRAINING = bool(RESTORED) and not RETRAIN and any((CKPT / p.name).exists() for p in CKPT.glob("*.pt")); print("finished runs found:", sorted(r["name"] for r in rows) or "none", "| predict-only mode:", SKIP_TRAINING)


In [ ]:
# ================= CELL 9 — LEXICAL BASELINE (BM25) AND HARD NEGATIVES  [RUNS NOW; CPU, a few minutes] =================
# BM25 is a baseline and the miner of HARD NEGATIVES for training (top-ranked candidates that are NOT relevant). Under RULE_MODE = "strict" it never decides the final ranking; it only appears in prediction_hybrid.json.
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse as sp
STOP = "the of and to in a is that be by for was as it on with or this not he which his are has an at from any have under were been had shall said such no but if may their they there its upon would other these those than also into made court section".split()
def bm25_matrix(D, k1=1.2, b=0.75, fitted=None):
    if fitted is None:
        cv = CountVectorizer(lowercase=True, token_pattern=r"(?u)\b[a-z0-9][a-z0-9]+\b", stop_words=STOP, dtype=np.float32, min_df=1); C = cv.fit_transform(D["c_text"]).tocsr(); Q = cv.transform(D["q_text"]).tocsr(); fitted = (C, Q)
    C, Q = fitted; N = C.shape[0]; dl = np.asarray(C.sum(1)).ravel(); avg = dl.mean() + 1e-9; dfreq = np.asarray((C > 0).sum(0)).ravel(); idf = np.log((N - dfreq + 0.5) / (dfreq + 0.5) + 1.0).astype(np.float32)
    Co = C.tocoo(); w = Co.data * (k1 + 1) / (Co.data + k1 * (1 - b + b * dl[Co.row] / avg)) * idf[Co.col]; W = sp.csr_matrix((w.astype(np.float32), (Co.row, Co.col)), shape=C.shape)
    Qw = Q.copy(); Qw.data = np.log1p(Qw.data); return np.asarray((Qw @ W.T).todense(), dtype=np.float32), fitted
t0 = time.time(); BM25, FIT = {}, {}
S0, FIT["dev"] = bm25_matrix(DATA["dev"]); grid = []
for k1 in (0.9, 1.2, 1.6):
    for b in (0.5, 0.75, 0.9):
        S, _ = bm25_matrix(DATA["dev"], k1, b, FIT["dev"]); grid.append((micro_at_k(rank_top(S, DATA["dev"]), DATA["dev"]["gold"])["f1"], k1, b))
BM_F1, BM_K1, BM_B = max(grid); print("BM25 dev grid (micro-F1@7, k1, b):", [(round(f, 4), k1, b) for f, k1, b in sorted(grid, reverse=True)[:4]])
for s in ("dev", "test", "train"): BM25[s], FIT[s] = bm25_matrix(DATA[s], BM_K1, BM_B, FIT.get(s))
m_bm = micro_at_k(rank_top(BM25["dev"], DATA["dev"]), DATA["dev"]["gold"]); print(f"BM25 (k1={BM_K1}, b={BM_B}) on dev: micro-F1@7 {m_bm['f1']:.4f} | P@7 {m_bm['p']:.4f} | R@7 {m_bm['r']:.4f} | {time.time() - t0:.0f}s")
HARD = []                                                                        # for every training query: the best-ranked NON-relevant training candidates (hard negatives)
for qi in range(len(DATA["train"]["q_ids"])):
    order = np.argsort(-BM25["train"][qi]); g = DATA["train"]["gold"][qi]; qself = DATA["train"]["c_pos"].get(DATA["train"]["q_ids"][qi]); HARD.append([int(c) for c in order[:60] if c not in g and c != qself][:40])
save_scores("B0_bm25", "bm25", "lexical baseline (not a Phase 1 model)", BM25["dev"], BM25["test"], m_bm, extra=dict(k1=BM_K1, b=BM_B), control=True)


In [ ]:
# ================= CELL 11 — CONTRASTIVE TRAINING (+ distillation)  [pure code; needs torch] =================
# Every step: B training queries; for each, one RELEVANT candidate and one BM25 HARD NEGATIVE from the training pool; every document is represented by K_TRAIN random chunks (mean of their vectors).
# Loss: InfoNCE over all 2B candidates of the batch (in-batch negatives + hard negatives), the other relevant candidates of a query masked out (they are not negatives). DISTILLATION adds the teacher's softmax over the same candidates (KL, temperature T).
TECH_MAP = {"full": "full", "partial": "partial", "lorma_plus": "lorma_plus", "ia3": "ia3", "kd": "full"}
DEFAULT_LR = {"full": 1e-4, "partial": 3e-4}                                              # the BiLSTM family: recurrent weights tolerate larger rates than the transformer
FAMILY = "bilstm"
def build_encoder(technique, hp=None, n_layers=None, n_prompt=20, init=None):
    bb, hidden, causal, ctx, fam = load_backbone_any(MODEL_PATHS[FAMILY]); assert fam == FAMILY
    if init == "dapt": bb.inner.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in torch.load(CKPT / "BL_dapt_inner.pt", map_location="cpu", weights_only=False).items()}, strict=True)       # the Phase 1 BiLSTM after continued pre-training
    model = DocEncoder(bb, hidden, n_prompt if technique == "prompt" else 0)
    if technique == "prompt":
        for p in model.backbone.parameters(): p.requires_grad = False
        model.prompt.requires_grad = True
    else: apply_method(model, TECH_MAP[technique], hp or {})
    return model
def trainable_state(model): names = {n for n, p in model.named_parameters() if p.requires_grad}; return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in names}
def load_retriever(name):
    """a scores-only VARIANT of a run (for example '<run>_c24', scored with more chunks) has no checkpoint of its own: it resolves to its base run's checkpoint and adapter settings; the meta returned is the requested run's own (its scoring mode, its dev score)"""
    meta = json.load(open(WORK / "runs" / name / "result.json")); base = meta.get("base_run", name); bmeta = json.load(open(WORK / "runs" / base / "result.json")) if base != name else meta
    model = build_encoder(bmeta["technique"], bmeta.get("hp"), bmeta.get("n_layers"), bmeta.get("n_prompt", 20), bmeta.get("init")); sd = torch.load(CKPT / f"{base}.pt", map_location="cpu", weights_only=False)
    miss = model.load_state_dict(sd, strict=False); assert not miss.unexpected_keys, miss.unexpected_keys[:3]; return model.to(device), meta
def base_meta(name): m = json.load(open(WORK / "runs" / name / "result.json")); return json.load(open(WORK / "runs" / m.get("base_run", name) / "result.json")) if m.get("base_run", name) != name else m
def train_retriever(name, technique, hp=None, minutes=30, lr=None, n_layers=None, teacher=None, kd_alpha=0.5, kd_T=2.0, B=8, K_TRAIN=2, tau=0.05, n_prompt=20, seed=SEED, note="", hard=None, n_neg=1, control=False, init=None):
    if run_done(name) and (CKPT / f"{name}.pt").exists(): r = json.load(open(WORK / "runs" / name / "result.json")); print(f"[{name}] already finished (restored): dev micro-F1@7 {r['dev_f1']:.4f}"); return r
    seed_all(seed); rng = np.random.default_rng(seed); t0 = time.time(); model = build_encoder(technique, hp, n_layers, n_prompt, init).to(device); lr = lr or DEFAULT_LR[technique]
    if technique in ("full", "partial", "kd") and device.type == "cuda" and hasattr(model.backbone.inner, "gradient_checkpointing_enable"): model.backbone.inner.gradient_checkpointing_enable()
    params = [p for p in model.parameters() if p.requires_grad]; n_tr, n_all = sum(p.numel() for p in params), sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01); scaler = torch.cuda.amp.GradScaler(enabled=AMP); Dt = DATA["train"]; TRQ = [q for q in range(len(Dt["q_ids"])) if Dt["gold"][q]]
    print(f"[{name}] {technique} | {n_all / 1e6:.1f}M params, {n_tr / 1e6:.2f}M trained ({100 * n_tr / n_all:.2f}%) | lr {lr} | {len(TRQ)} training queries | B={B}, {K_TRAIN} chunks per document | cap {minutes} min" + (f" | teacher {teacher[1]}" if teacher else ""))
    def pick(chs): return [chs[i] for i in rng.choice(len(chs), size=min(K_TRAIN, len(chs)), replace=False)]
    def doc_vecs(m, docs):
        flat = [c for d in docs for c in d]; ids, mask = pad_chunks(flat)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP): v = m(ids.to(device), mask.to(device))
        out, j = [], 0
        for d in docs: out.append(F.normalize(v[j:j + len(d)].float().mean(0), dim=-1)); j += len(d)
        return torch.stack(out)
    est_steps, step, losses, warm = None, 0, [], 30; model.train()
    while True:
        el = (time.time() - t0) / 60
        if el >= minutes * 0.95: break
        qs = rng.choice(TRQ, size=min(B, len(TRQ)), replace=False); pos = [int(rng.choice(sorted(Dt["gold"][q]))) for q in qs]; HH = hard if hard is not None else HARD; neg = [int(x) for q in qs for x in (rng.choice(HH[q], size=min(n_neg, len(HH[q])), replace=False) if HH[q] else rng.integers(0, len(Dt["c_ids"]), size=n_neg))]
        cands = pos + neg; qd = [pick(Dt["q_chunks"][q]) for q in qs]; cd = [pick(Dt["c_chunks"][c]) for c in cands]
        mask_ = torch.zeros(len(qs), len(cands), dtype=torch.bool)
        for i, q in enumerate(qs):
            for j, c in enumerate(cands):
                if j != i and c in Dt["gold"][q]: mask_[i, j] = True                # another relevant candidate of the same query is not a negative
        qv, cv = doc_vecs(model, qd), doc_vecs(model, cd); logits = (qv @ cv.T / tau).masked_fill(mask_.to(device), -1e4); tgt = torch.arange(len(qs), device=device); loss = F.cross_entropy(logits, tgt)
        if teacher is not None:
            with torch.no_grad(): tl = (doc_vecs(teacher[0], qd) @ doc_vecs(teacher[0], cd).T / tau).masked_fill(mask_.to(device), -1e4)
            loss = kd_alpha * loss + (1 - kd_alpha) * kd_T ** 2 * F.kl_div(F.log_softmax(logits / kd_T, -1), F.softmax(tl / kd_T, -1), reduction="batchmean")
        lr_now = lr * min(1.0, (step + 1) / warm); [g.__setitem__("lr", lr_now) for g in opt.param_groups]
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update(); step += 1; losses.append(float(loss))
        if step % 50 == 0: print(f"   step {step} | loss {np.mean(losses[-50:]):.4f} | {el:.1f} min", flush=True)
    train_min = (time.time() - t0) / 60; print(f"   trained {step} steps in {train_min:.1f} min (last-50 loss {np.mean(losses[-50:]) if losses else float('nan'):.4f})")
    if hasattr(model.backbone.inner, "gradient_checkpointing_disable"): model.backbone.inner.gradient_checkpointing_disable()
    torch.save(trainable_state(model), CKPT / f"{name}.pt"); mem = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else float("nan")
    t1 = time.time(); m = evaluate_model(model, name, technique, note, extra=dict(hp=hp, n_layers=n_layers, n_prompt=n_prompt if technique == "prompt" else None, lr=lr, steps=step, train_minutes=train_min, params_M=n_all / 1e6, trainable_M=n_tr / 1e6, peak_mem_gb=mem, teacher=teacher[1] if teacher else None, n_neg=n_neg, init=init, hard_negatives="round 2 (mined by a trained model)" if hard is not None else "BM25"), control=control)
    r = json.load(open(WORK / "runs" / name / "result.json")); del model, opt; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    return r


In [ ]:
# ================= CELL 14 — SETTINGS, TIME BUDGET, HELPERS  [pure code] =================
# The Phase 7 task does not require every Phase 1 architecture, but the BiLSTM is the fourth one: this notebook implements the BiLSTM retriever (a bi-encoder: every 512-token chunk -> the mean of the Phase 1 BiLSTM's forward and backward states -> a unit vector),
# trains it with the same contrastive recipe as the ModernBERT runs (in-batch + BM25 hard negatives), stores it in a new Kaggle dataset and generates the predictions. The BiLSTM was only lightly pre-trained in Phase 1 (validation loss 5.96 against a word-frequency baseline of 6.56),
# so do not expect it to match the transformer; its value is the fourth architecture, the transfer analysis and, possibly, diversity for an ensemble.
TOTAL_BUDGET_MIN, FINAL_RESERVE_MIN = (330, 15) if TIME_PROFILE == "normal" else (170, 10)
CAPS = dict(DAPT=40, BL1=30, BL3=30, BL2=30, BL4=30) if TIME_PROFILE == "normal" else dict(DAPT=20, BL1=15, BL3=15, BL2=15, BL4=15)
MIN_GAIN = 0.005
DAPT_SANITY_MAX = 0.85 * math.log(32001)                                                  # the Phase 1 LM loss is about 6; a loss near ln(vocab) = 10.4 would mean the loss formula does not match the checkpoint
if SELFTEST: TOTAL_BUDGET_MIN, CAPS, MIN_GAIN, DAPT_SANITY_MAX = 999, {k: 0.15 for k in CAPS}, 0.0, 99.0                 # a tiny random checkpoint has no pre-trained loss to check
def left_min(): return TOTAL_BUDGET_MIN - (time.time() - NB_T0) / 60
def can_run(label, need):
    ok = left_min() - FINAL_RESERVE_MIN >= need
    if not ok: print(f"SKIPPED {label}: needs about {need:.0f} min but only {left_min() - FINAL_RESERVE_MIN:.0f} min of the {TOTAL_BUDGET_MIN}-minute budget remain")
    return ok
def z(S): S = S.astype(np.float32); return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-6)
def load_sc(n, s, root=None): return np.load((root or WORK) / "runs" / n / f"{s}_scores.npy").astype(np.float32)
def f1_of(S, s="dev"): return micro_at_k(rank_top(S, DATA[s]), DATA[s]["gold"])
def passes(S_new, S_ref, min_gain=None):
    m_new, m_ref = f1_of(S_new), f1_of(S_ref); d, lo, hi = boot_diff(m_new, m_ref); return (m_new["f1"] - m_ref["f1"] >= (MIN_GAIN if min_gain is None else min_gain) and lo > 0), m_new["f1"] - m_ref["f1"], lo, hi
def ok_adopt(a): return not (a is False or str(a) == "False")
assert "bilstm" in MODEL_PATHS or SKIP_TRAINING, "the Phase 1 BiLSTM checkpoint was not found: attach the dataset cs781-phase1-v3-bilstm-recurrent (it holds final_model.pt), or attach the retriever dataset / an earlier output of this notebook for predict-only mode"
print(f"budget {TOTAL_BUDGET_MIN} min ({(time.time() - NB_T0) / 60:.0f} used) | caps {CAPS} | DAPT {'on' if DAPT_ON else 'off'} | upload {'on' if UPLOAD_DATASET else 'off'} | predict-only mode: {SKIP_TRAINING} | BiLSTM checkpoint: {MODEL_PATHS.get('bilstm', 'not attached (predict-only)')}")


In [ ]:
# ================= CELL 15 — THE PHASE 1 BiLSTM AS IT IS (zero-shot retriever)  [NEEDS the BiLSTM Phase 1 checkpoint + a GPU; about 8-12 min] =================
# The untouched Phase 1 BiLSTM as a retriever: how well do its representations transfer to prior-case retrieval without any retrieval training? Also the first prediction file and the measured cost of one evaluation.
if not SKIP_TRAINING and not run_done("Z0_bilstm_zero_shot"):
    def z0():
        bb, hidden, *_ = load_backbone_any(MODEL_PATHS["bilstm"]); m = DocEncoder(bb, hidden).to(device); n = sum(p.numel() for p in m.parameters()); evaluate_model(m, "Z0_bilstm_zero_shot", "zero-shot", "Phase 1 BiLSTM, no retrieval training", extra=dict(params_M=n / 1e6, trainable_M=0.0), control=True)
        del m; gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
    safe("Z0", z0)
EVAL_MIN = float(json.load(open(WORK / "runs" / "Z0_bilstm_zero_shot" / "result.json")).get("eval_minutes", 8)) if (WORK / "runs" / "Z0_bilstm_zero_shot" / "result.json").exists() else 8.0
update_final(); print(f"one evaluation (dev + test, both scoring modes) takes about {EVAL_MIN:.1f} min")


In [ ]:
# ================= CELL 16 — CONTINUED PRE-TRAINING OF THE PHASE 1 BiLSTM ON THE PHASE 7 TRAIN DOCUMENTS (optional)  [NEEDS the BiLSTM Phase 1 checkpoint + a GPU; capped at CAPS DAPT min] =================
# The Phase 1 BiLSTM saw only 560 steps. Here it continues its OWN language-model objective (forward LM + backward LM, exactly the Phase 1 loss: the same loss on packed 512-token blocks) on the TRAIN documents (queries and candidates) of Phase 7: the provided text, no external corpus, no dev or test text.
# (arch_loaders.BiLSTMLM is the DOWNSTREAM loader and has no forward(): the Phase 1 loss is re-implemented below from the checkpoint's own layers, line for line as in the Phase 1 notebook.)
# The result is a different STARTING POINT for the retrieval runs (BL3/BL4); it is used in the final system only if it wins on dev. Sanity check first: the loss at the start must look like the Phase 1 loss (about 6), otherwise the formula does not match the checkpoint and the step is skipped.
DAPT_INFO = {}
def bilstm_lm_loss(inner, x):
    """the Phase 1 pre-training loss: the forward LSTM predicts the next token, the backward LSTM (run over the flipped block) predicts the previous one; the output layer is tied to the embedding; packed blocks, no padding"""
    xb = torch.flip(x, [1]); hf, _ = inner.fwd(inner.emb(x)); hb, _ = inner.bwd(inner.emb(xb)); E, V_ = inner.emb.weight, inner.emb.weight.shape[0]
    def lm(h, t): return F.cross_entropy((h @ E.t() + inner.out_bias).float().reshape(-1, V_), t.reshape(-1))
    return 0.5 * (lm(inner.proj_f(inner.norm_f(hf[:, :-1])), x[:, 1:]) + lm(inner.proj_b(inner.norm_b(hb[:, :-1])), xb[:, 1:]))
if DAPT_ON and not SKIP_TRAINING and not (CKPT / "BL_dapt_inner.pt").exists() and can_run("continued pre-training", CAPS["DAPT"] + 3):
    def dapt():
        seed_all(SEED); rng = np.random.default_rng(SEED); L, B = 512, 12
        def stream(split): return np.concatenate([np.concatenate([np.concatenate([d, [EOS if EOS is not None else 0]]) for d in DATA[split][f"{side}_tok"]]) for side in ("q", "c")]).astype(np.int64)
        tr, dv = stream("train"), stream("dev"); L = min(L, len(dv) - 2, len(tr) - 2)
        bb, hidden, *_ = load_backbone_any(MODEL_PATHS["bilstm"]); inner = bb.inner.to(device)
        def blocks(s, n, g): starts = g.integers(0, len(s) - L - 1, n); return torch.from_numpy(np.stack([s[a:a + L] for a in starts])).to(device)
        @torch.no_grad()
        def held_out():
            inner.eval(); g = np.random.default_rng(123); out = []
            for _ in range(6):
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP): out.append(float(bilstm_lm_loss(inner, blocks(dv, 8, g))))
            inner.train(); return float(np.mean(out))
        l0 = held_out(); print(f"language-model loss on held-out DEV documents before continued pre-training: {l0:.3f} (the Phase 1 validation loss was about 5.96; ln(vocabulary) = {math.log(VOCAB):.2f})")
        if l0 > DAPT_SANITY_MAX: print(f"SKIPPED continued pre-training: the starting loss {l0:.2f} is too high for a pre-trained BiLSTM (limit {DAPT_SANITY_MAX:.2f}): the loss formula does not match this checkpoint"); return
        params = [p for p in inner.parameters() if p.requires_grad]; opt = torch.optim.AdamW(params, lr=2e-4, weight_decay=0.01); scaler = torch.cuda.amp.GradScaler(enabled=AMP); t0 = time.time(); step, losses = 0, []; inner.train(); total = CAPS["DAPT"] * 0.95 * 60
        while time.time() - t0 < total:
            frac = (time.time() - t0) / total; lr = 2e-4 * min(1.0, (step + 1) / 30) * (1 - 0.9 * frac)
            for g_ in opt.param_groups: g_["lr"] = lr
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP): loss = bilstm_lm_loss(inner, blocks(tr, B, rng))
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update(); step += 1; losses.append(float(loss))
            if step % 100 == 0: print(f"   step {step} | train loss {np.mean(losses[-100:]):.3f} | {(time.time() - t0) / 60:.1f} min", flush=True)
        l1 = held_out(); print(f"continued pre-training: {step} steps ({step * B * L / 1e6:.1f}M tokens) in {(time.time() - t0) / 60:.1f} min | held-out DEV language-model loss {l0:.3f} -> {l1:.3f}")
        DAPT_INFO.update(l0=l0, l1=l1, steps=step, minutes=(time.time() - t0) / 60, tokens_M=step * B * L / 1e6)
        if l1 < l0: torch.save({k: (v.half() if v.is_floating_point() else v).cpu() for k, v in inner.state_dict().items()}, CKPT / "BL_dapt_inner.pt"); json.dump(DAPT_INFO, open(WORK / "dapt_info.json", "w"))
        else: print("the held-out loss did not fall: the continued pre-training is discarded")
        del inner, bb, opt; gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
    safe("continued pre-training", dapt)
HAS_DAPT = (CKPT / "BL_dapt_inner.pt").exists(); print("continued-pretrained BiLSTM available:", HAS_DAPT)


In [ ]:
# ================= CELL 17 — THE BiLSTM RETRIEVERS  [NEEDS the BiLSTM Phase 1 checkpoint + a GPU; each capped, each checks the time budget, each updates prediction.json when it finishes] =================
# Adaptation of the BiLSTM: LoRA, LoRMA, IA3 and the other adapters act on linear layers and an LSTM has none, so the techniques are PARTIAL fine-tuning (the top half of the layers; the embeddings and the lower layers frozen) and FULL fine-tuning, each from the Phase 1 weights
# and, if continued pre-training worked, from those weights as a second starting point. BL1 partial (Phase 1) -> BL3 partial (continued pre-training) -> BL2 full (Phase 1) -> BL4 full (continued pre-training).
PLAN = [("BL1", "BL1_bilstm_partial", "partial", dict(frac=0.5), None, "partial fine-tuning: the top half of the LSTM layers, from the Phase 1 weights"), ("BL3", "BL3_bilstm_partial_dapt", "partial", dict(frac=0.5), "dapt", "partial fine-tuning from the continued-pretrained BiLSTM"),
        ("BL2", "BL2_bilstm_full", "full", None, None, "full fine-tuning from the Phase 1 weights"), ("BL4", "BL4_bilstm_full_dapt", "full", None, "dapt", "full fine-tuning from the continued-pretrained BiLSTM")]
if not SKIP_TRAINING:
    for key, name, tech, hp, init, note in PLAN:
        if init == "dapt" and not HAS_DAPT: continue
        if can_run(name, CAPS[key] + EVAL_MIN + 2): safe(name, lambda: train_retriever(name, tech, hp, minutes=CAPS[key], init=init, note=note, control=(key in ("BL3", "BL4"))))
update_final(); T_ = runs_table(); print("BiLSTM runs so far:", [(r["name"], round(r["dev_f1"], 4)) for r in T_.sort_values("dev_f1", ascending=False).to_dict("records") if str(r["name"]).startswith(("BL", "Z0"))])


In [ ]:
# ================= CELL 18 — BiLSTM ENSEMBLE AND FUSION WITH AN EARLIER MODERNBERT SYSTEM  [RUNS NOW; CPU; seconds] =================
# (1) The BiLSTM runs are averaged (per-query standardised scores; the largest ensemble within 0.003 of the best dev score). (2) If the output of the ModernBERT notebooks (Parts 1/2, folder pcr/) is attached, its final system's scores are fused with the BiLSTM's: the BiLSTM is another Phase 1 architecture and
# may add diversity; the weight is tuned on dev (one parameter) and the fusion is adopted only if dev micro-F1@7 rises by MIN_GAIN with a paired 95% interval above 0. Otherwise it is kept in the table, flagged, and never becomes prediction.json.
def bl_runs(): T = runs_table(); return list(T[T.name.str.startswith("BL") & T.technique.isin(["full", "partial"])].sort_values("dev_f1", ascending=False).name) if len(T) else []
BLN = bl_runs(); PRIOR = None
if len(BLN) >= 2:
    ks = sorted({k for k in (1, 2, len(BLN)) if 1 <= k <= len(BLN)}); res = {k: f1_of(sum(z(load_sc(n, "dev")) for n in BLN[:k]) / k)["f1"] for k in ks}; kbest = max(k for k, f in res.items() if f >= max(res.values()) - 0.003)
    if kbest >= 2:
        Sd, St = [sum(z(load_sc(n, s)) for n in BLN[:kbest]) / kbest for s in ("dev", "test")]; save_scores("N_bilstm_ens", "ensemble", f"mean of standardised scores of {BLN[:kbest]}", Sd, St, f1_of(Sd), extra=dict(members=BLN[:kbest]), control=True); print(f"BiLSTM ensemble sizes on dev {({k: round(f, 4) for k, f in res.items()})} -> top {kbest}: dev micro-F1@7 {f1_of(Sd)['f1']:.4f}")
for src in sorted({Path(p).parent for p in glob.glob("/kaggle/input/**/pcr/ledger.csv", recursive=True)} - {WORK}):
    fc = src / "final_choice.json"
    if not fc.exists(): continue
    sysname = json.load(open(fc))["system"]; d, t = src / "runs" / sysname / "dev_scores.npy", src / "runs" / sysname / "test_scores.npy"
    if d.exists() and t.exists() and np.load(d).shape == (len(DATA["dev"]["q_ids"]), len(DATA["dev"]["c_ids"])) and np.load(t).shape == (len(DATA["test"]["q_ids"]), len(DATA["test"]["c_ids"])): PRIOR = dict(name=sysname, dev=np.load(d).astype(np.float32), test=np.load(t).astype(np.float32), src=str(src)); break
if PRIOR and (BLN or (WORK / "runs" / "Z0_bilstm_zero_shot").exists()):
    best_bl = (["N_bilstm_ens"] if (WORK / "runs" / "N_bilstm_ens").exists() else []) + BLN; bn = best_bl[0] if best_bl else "Z0_bilstm_zero_shot"; Bd, Bt = z(load_sc(bn, "dev")), z(load_sc(bn, "test")); Pd, Pt = z(PRIOR["dev"]), z(PRIOR["test"])
    w = max((0.1, 0.2, 0.35, 0.5, 1.0), key=lambda w_: f1_of(Pd + w_ * Bd)["f1"]); Fd, Ft = Pd + w * Bd, Pt + w * Bt; ok, d, lo, hi = passes(Fd, Pd); print(f"earlier ModernBERT system {PRIOR['name']}: dev micro-F1@7 {f1_of(Pd)['f1']:.4f} | + {w} x {bn}: {f1_of(Fd)['f1']:.4f} ({d:+.4f} [{lo:+.4f}, {hi:+.4f}]) -> {'ADOPTED' if ok else 'not adopted'}")
    save_scores("X_prior_plus_bilstm", "ensemble", f"earlier ModernBERT system {PRIOR['name']} + {w} x {bn} (Phase 1 models only)", Fd, Ft, f1_of(Fd), extra=dict(w=w, prior=PRIOR["name"], bilstm=bn), control=True, adopted=ok)
elif PRIOR is None: print("no earlier ModernBERT output (pcr/) attached: no fusion; attach the output of Part 1/2 to test whether the BiLSTM adds anything to the transformer")


In [ ]:
# ================= CELL 19 — CREATE THE KAGGLE DATASET THAT STORES THE MODEL  [NEEDS the KAGGLE_USERNAME / KAGGLE_KEY secrets + Internet; a few minutes] =================
# Files are stored FLAT (a Kaggle dataset created from a folder skips sub-folders): the trained retriever checkpoints (only the trained parameters of a partial run; the full state of the continued-pretrained BiLSTM in half precision), every run's result and score matrices,
# arch_loaders.py (the Phase 1 loader that builds the BiLSTM), a manifest with the file sizes, a README. Attached to a later run, it makes this notebook predict-only: nothing is trained again. The upload is VERIFIED (the CLI can print an error and still exit 0): the dataset status is polled.
import subprocess
def publish_dataset(stage, owner, slug, title, poll=12, wait=15):
    json.dump(dict(title=title, id=f"{owner}/{slug}", licenses=[{"name": "other"}]), open(stage / "dataset-metadata.json", "w")); BAD = r"(?i)creation error|slugs? .*null|was not created|failed|already (exists|in use)|\berror\b"; clean = lambda t: "\n".join(l for l in t.splitlines() if "%|" not in l and l.strip())
    r = subprocess.run(["kaggle", "datasets", "create", "-p", str(stage), "-r", "skip"], capture_output=True, text=True); o = (r.stdout + r.stderr).strip(); print(clean(o)[-700:])
    if r.returncode != 0 or re.search(BAD, o):
        print("-> the create step reported a problem; trying a new VERSION (the dataset may already exist)"); r = subprocess.run(["kaggle", "datasets", "version", "-p", str(stage), "-m", "BiLSTM retriever update", "-r", "skip"], capture_output=True, text=True); print(clean(r.stdout + r.stderr)[-700:])
    for _ in range(poll):
        st = subprocess.run(["kaggle", "datasets", "status", f"{owner}/{slug}"], capture_output=True, text=True); out = (st.stdout + st.stderr).strip()
        if st.returncode == 0 and "ready" in out.lower(): return True, out
        time.sleep(wait if not SELFTEST else 0.1)
    return False, out
DS_OK, DS_NOTE = None, "not attempted"
done = [n for n in sorted(p.name for p in (WORK / "runs").iterdir()) if n.startswith(("BL", "Z0", "N_bilstm", "X_prior", "B0", "B1"))] if (WORK / "runs").exists() else []
if UPLOAD_DATASET and not SKIP_TRAINING and any(n.startswith("BL") for n in done):
    owner = os.environ.get("KAGGLE_USERNAME"); assert owner and os.environ.get("KAGGLE_KEY"), "the Kaggle secrets are missing: attach KAGGLE_USERNAME and KAGGLE_KEY (or set UPLOAD_DATASET = False)"
    stage = WORK / "dataset_stage"; shutil.rmtree(stage, ignore_errors=True); stage.mkdir(); files = {}
    for run in done:
        for fn in RUN_FILES:
            f = WORK / "runs" / run / fn
            if f.exists(): shutil.copy2(f, stage / f"{run}__{fn}"); files[f"{run}__{fn}"] = dict(bytes=(stage / f"{run}__{fn}").stat().st_size)
        if run.startswith("BL") and (CKPT / f"{run}.pt").exists(): shutil.copy2(CKPT / f"{run}.pt", stage / f"ckpt__{run}.pt"); files[f"ckpt__{run}.pt"] = dict(bytes=(stage / f"ckpt__{run}.pt").stat().st_size)
    if (CKPT / "BL_dapt_inner.pt").exists(): shutil.copy2(CKPT / "BL_dapt_inner.pt", stage / "ckpt__BL_dapt_inner.pt"); files["ckpt__BL_dapt_inner.pt"] = dict(bytes=(stage / "ckpt__BL_dapt_inner.pt").stat().st_size)
    try: shutil.copy2(sys.modules["arch_loaders"].__file__, stage / "arch_loaders.py"); files["arch_loaders.py"] = dict(bytes=(stage / "arch_loaders.py").stat().st_size)
    except Exception as e: print("arch_loaders.py not copied:", e)
    T_ = runs_table(); (stage / "results.json").write_text(json.dumps([{k: v for k, v in r.items() if not isinstance(v, (dict, list))} for r in T_.to_dict("records")], default=str, indent=1)); files["results.json"] = dict(bytes=(stage / "results.json").stat().st_size)
    (stage / "README.md").write_text("# CS781 Phase 7 BiLSTM retriever\n\nFlat files: `ckpt__<run>.pt` = trained parameters of a BiLSTM retriever (a bi-encoder on the Phase 1 BiLSTM: mean of the forward and backward states of 512-token chunks); `ckpt__BL_dapt_inner.pt` = the Phase 1 BiLSTM after continued pre-training (half precision); `<run>__result.json`, `<run>__dev_scores.npy`, `<run>__test_scores.npy` = per-run results and score matrices (queries x the split's own candidate pool); `arch_loaders.py` = the Phase 1 loader.\n"
        "Attach this dataset (and the Phase 1 BiLSTM dataset) to the Phase 7 BiLSTM notebook with RETRAIN = False: it restores everything and only regenerates the predictions.\n"); files["README.md"] = dict(bytes=(stage / "README.md").stat().st_size)
    json.dump(dict(files=files, runs=done), open(stage / "BILSTM_RETRIEVER_MANIFEST.json", "w")); tot = sum(v["bytes"] for v in files.values()) / 2 ** 20; print(f"staged {len(files) + 1} files, {tot:.0f} MiB: {sorted(files)[:8]}...")
    DS_OK, DS_NOTE = publish_dataset(stage, owner, DATASET_SLUG, DATASET_TITLE)
    print(f"\nKaggle dataset {owner}/{DATASET_SLUG}: " + ("CREATED and READY (verified with the dataset status)" if DS_OK else f"NOT VERIFIED as ready yet ({DS_NOTE[:90]!r}): check https://www.kaggle.com/datasets/{owner}/{DATASET_SLUG}; the staged folder {stage} and this notebook's output hold every file")); print("to reuse it: Add Input > your dataset, then run with RETRAIN = False (predict-only)")
else: print("dataset creation skipped:", "predict-only mode (a retriever dataset or earlier output is attached)" if SKIP_TRAINING else "UPLOAD_DATASET = False" if not UPLOAD_DATASET else "no BiLSTM run finished")


In [ ]:
# ================= CELL 20 — GENERATE THE PREDICTIONS  [predict-only with REGENERATE_FROM_CHECKPOINT needs a GPU, about 8-12 min; otherwise seconds] =================
# prediction_bilstm_only.json (+ .zip): the best BiLSTM-based system on dev, 7 unique candidates per test query from the TEST pool. prediction.json (+ .zip) = the best ADOPTED rule-safe system of this notebook (the BiLSTM alone, or its fusion with the attached ModernBERT system if that was adopted).
# In predict-only mode the saved checkpoint is loaded and the test pool is encoded again: the new scores are compared with the stored ones (a check that the dataset reproduces the model).
T = runs_table(); BLS = T[T.name.str.startswith(("BL", "Z0_bilstm", "N_bilstm")) & ~T.technique.isin(["bm25"])].sort_values("dev_f1", ascending=False); assert len(BLS), "no BiLSTM system exists: nothing to generate (attach the BiLSTM Phase 1 dataset or the retriever dataset)"
best_bl = BLS.iloc[0]["name"]; S_test = load_sc(best_bl, "test"); REGEN_NOTE = "scores as stored"
if SKIP_TRAINING and REGENERATE_FROM_CHECKPOINT and (CKPT / f"{best_bl}.pt").exists() and best_bl.startswith("BL"):
    def regen():
        global S_test, REGEN_NOTE
        model, meta = load_retriever(best_bl); mode = meta.get("mode", "maxsim"); S_new = score(*encode_docs(model, chunks_for("test", "q", int(meta.get("chunks") or MAX_CHUNKS))), *encode_docs(model, chunks_for("test", "c", int(meta.get("chunks") or MAX_CHUNKS))), mode)
        a, b = rank_top(S_new, DATA["test"]), rank_top(S_test, DATA["test"]); agree = float(np.mean([len(set(x.tolist()) & set(y.tolist())) / TOP_K for x, y in zip(a, b)])); diff = float(np.abs(z(S_new) - z(S_test)).max()); S_test = S_new
        REGEN_NOTE = f"re-encoded from the saved checkpoint {best_bl}: top-7 overlap with the stored scores {agree:.1%}, largest standardised-score difference {diff:.4f}"; print(REGEN_NOTE); del model; gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
    safe("regenerate from the saved checkpoint", regen)
BL_PATH = PRED_PATH.with_name("prediction_bilstm_only.json"); write_prediction(BL_PATH, rank_top(S_test, DATA["test"]), DATA["test"]); validate_prediction(BL_PATH, DATA["test"]); zip_prediction(BL_PATH, BL_PATH.with_suffix(".zip"))
FINAL = update_final(verbose=False); print(f"prediction_bilstm_only.json <- {best_bl} (dev micro-F1@7 {float(BLS.iloc[0]['dev_f1']):.4f}) | {REGEN_NOTE} | validated: {len(DATA['test']['q_ids'])} test queries x {TOP_K} unique candidates from the TEST pool")
fin_f = float(runs_table().set_index("name").loc[FINAL, "dev_f1"]); print(f"prediction.json <- {FINAL} (dev micro-F1@7 {fin_f:.4f})" + ("" if FINAL == best_bl else "  [a fusion with the earlier ModernBERT system, adopted by the dev rule]")); print("first entry:", list(json.load(open(BL_PATH)).items())[0])


In [ ]:
# ================= CELL 21 — COMPARISON, REPORT AND OUTPUT CHECK  [RUNS NOW] =================
T = runs_table().sort_values("dev_f1", ascending=False); ref = {r["name"]: f1_of(load_sc(r["name"], "dev")) for r in T.to_dict("records")}; rows = []
for r in T.to_dict("records"):
    d1, lo, hi = boot_diff(ref[r["name"]], ref["B0_bm25"]) if "B0_bm25" in ref and r["name"] != "B0_bm25" else (0.0, 0.0, 0.0)
    rows.append(dict(system=r["name"], technique=r["technique"], init=("continued pre-training" if r.get("init") == "dapt" else "Phase 1 weights") if str(r["name"]).startswith("BL") else "", adopted=ok_adopt(r.get("adopted", True)), dev_f1=round(r["dev_f1"], 4), dev_p=round(r["dev_p"], 4), dev_r=round(r["dev_r"], 4), vs_bm25=f"{d1:+.4f} [{lo:+.4f}, {hi:+.4f}]", mode=r.get("mode"), trainable_M=r.get("trainable_M"), train_min=None if r.get("train_minutes") is None else round(float(r["train_minutes"]), 1)))
COMP = pd.DataFrame(rows); COMP.to_csv(WORK / "comparison.csv", index=False); print("COMPARISON ON DEV (micro-F1@7; intervals: paired bootstrap over queries against unigram BM25):"); print(COMP.to_string(index=False))
info = json.load(open(WORK / "dapt_info.json")) if (WORK / "dapt_info.json").exists() else {}; z0 = float(T[T.name == "Z0_bilstm_zero_shot"].dev_f1.iloc[0]) if (T.name == "Z0_bilstm_zero_shot").any() else float("nan"); bls = T[T.name.str.startswith("BL")]
rep = ["# Phase 7: the BiLSTM retriever (dev numbers; not official test scores)", "", f"- the Phase 1 BiLSTM as it is: dev micro-F1@7 {z0:.4f}", f"- best BiLSTM retriever: {bls.iloc[0]['name'] if len(bls) else 'none'} (dev micro-F1@7 {float(bls.iloc[0]['dev_f1']) if len(bls) else float('nan'):.4f})",
       f"- continued pre-training on the train documents: {'language-model loss ' + format(info['l0'], '.3f') + ' -> ' + format(info['l1'], '.3f') + ' on held-out dev documents (' + format(info['steps']) + ' steps)' if info else 'not run or discarded'}", f"- Kaggle dataset: {DATASET_SLUG} ({'verified ready' if DS_OK else 'not created or not verified'})", f"- predictions: prediction_bilstm_only.json (BiLSTM alone) and prediction.json ({FINAL})", "",
       "A BiLSTM trained on 827 queries from a lightly pre-trained Phase 1 model is not expected to match the ModernBERT bi-encoder; use prediction.json of the ModernBERT notebooks unless this notebook's dev score says otherwise.", "Dev numbers are local evaluation, not official test scores."]
(WORK / "analysis.md").write_text("\n".join(rep), encoding="utf-8"); print("\n" + "\n".join(rep)); shutil.rmtree(WORK / "hf", ignore_errors=True)
for p in (PRED_PATH, BL_PATH): validate_prediction(p, DATA["test"]); print(f"{p.name}: VALID ({len(DATA['test']['q_ids'])} test queries x {TOP_K} unique candidates from the test pool) | zip exists: {p.with_suffix('.zip').exists()}")
print("outputs:", sorted(p.name for p in WORK.iterdir())[:20])
